# Manipolare stringhe

## Esercizio

Implementare la codifica [ROT13](https://it.wikipedia.org/wiki/ROT13) in Python. 

La codifica ROT13 è un semplice cifrario a sostituzione che sostituisce ogni lettera con la lettera 13 posizioni dopo di essa nell'alfabeto. Ad esempio, 'A' diventa 'N', 'B' diventa 'O', e così via. Se si raggiunge la fine dell'alfabeto, si ricomincia dall'inizio.

### Soluzione

Come mostrato, per passare da una `lettera` alla sua posizione rispetto alla `A` si può usare `ord(lettera) - ord('A')`, e viceversa per passare da una `posizione` alla lettera corrispondente si può usare `chr(posizione + ord('A'))`.

Inoltre, per "ricominciare dall'inizio" la nuova posizione può essere calcolata come `(posizione + 13) % 26`, dove `26` è il numero di lettere nell'alfabeto.

In [40]:
def rot13(testo):

  def _rot13(lettera):
    if not lettera.isalpha(): return lettera
    posizione = ord(lettera) - ord('A')
    return chr((posizione + 13) % 26 + ord('A'))

  return ''.join(_rot13(lettera) for lettera in testo.upper())

In [44]:
testo = 'La scena in cui il personaggio principale, detto Puzzone, muore non mi piace'
offuscato = rot13(testo)

offuscato

'YN FPRAN VA PHV VY CREFBANTTVB CEVAPVCNYR, QRGGB CHMMBAR, ZHBER ABA ZV CVNPR'

In [45]:
rot13(offuscato)

'LA SCENA IN CUI IL PERSONAGGIO PRINCIPALE, DETTO PUZZONE, MUORE NON MI PIACE'

# Il modulo `re` per le espressioni regolari

Sulla falsariga di [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) e [Regular expression operations](https://docs.python.org/3/library/re.html) dalla documentazione ufficiale.

In [1]:
import re

## Uso di base

In [2]:
# raw string (https://docs.python.org/3/reference/lexical_analysis.html#string-and-bytes-literals)

print(r'a\nb')

a\nb


In [3]:
# uso diretto

re.match(r'a|b', 'a')

<re.Match object; span=(0, 1), match='a'>

In [4]:
# o pre-compilando il pattern

p = re.compile(r'a|b')

p.match('b')

<re.Match object; span=(0, 1), match='b'>

In [5]:
# differenza fullmatch/match/search

p.fullmatch('xa'), p.fullmatch('ay')

(None, None)

In [6]:
p.match('xa'), p.match('ay')

(None, <re.Match object; span=(0, 1), match='a'>)

In [7]:
p.search('xay')

<re.Match object; span=(1, 2), match='a'>

In [8]:
# tutti?

list(p.finditer('mamma bella'))

[<re.Match object; span=(1, 2), match='a'>,
 <re.Match object; span=(4, 5), match='a'>,
 <re.Match object; span=(6, 7), match='b'>,
 <re.Match object; span=(10, 11), match='a'>]

In [9]:
# ma anche più semplicemente

p.findall('banana')

['b', 'a', 'a', 'a']

## Accedere alle sottostringhe

In [10]:
prefix = '02'
number = '342573'
telephone = prefix + '/' + number

In [11]:
# gruppi "semplici"

In [12]:
p = re.compile(r'([0-9]+)/([0-9]+)')

In [13]:
m = p.match(telephone)
m.groups()

('02', '342573')

In [14]:
# gruppi "annidati"

p = re.compile(r'(([0-9]+)/)?([0-9]+)')

In [15]:
m0, m1 = p.match(telephone), p.match(number)
m0.groups(), m1.groups()

(('02/', '02', '342573'), (None, None, '342573'))

In [16]:
# gruppi "denominati"

p = re.compile(r'((?P<prefix>[0-9]+)/)?(?P<number>[0-9]+)')

In [17]:
m0, m1 = p.match(telephone), p.match(number)
m0.groupdict(), m1.groupdict()

({'prefix': '02', 'number': '342573'}, {'prefix': None, 'number': '342573'})

In [18]:
# gruppi senza cattura

p = re.compile(r'(?:([0-9]+)/)?([0-9]+)')

In [19]:
m0, m1 = p.match(telephone), p.match(number)
m0.groups(), m1.groups()

(('02', '342573'), (None, '342573'))

## Metacaratteri e flags

In [20]:
# . ^ $ * + ? { } [ ] \ | ( )

In [21]:
# set negati

re.findall(r'[^aeiou]', 'just consonants')

['j', 's', 't', ' ', 'c', 'n', 's', 'n', 'n', 't', 's']

In [22]:
# multiline

text = """I've seen things you people wouldn't believe
Attack ships on fire off the shoulder of Orion
I watched C-beams glitter in the dark near the Tannhäuser Gate
All those moments will be lost in time, like tears in rain
Time to die"""

In [23]:
# dotall rende il . (jolly) in grado di catturare anche l'a-capo

'believe\nAttack' in re.findall(r'\S+.\S+', text, re.DOTALL)

True

In [24]:
'believe\nAttack' in re.findall(r'\S+.\S+', text)

False

In [25]:
# multiline permette a ^ e $ (inizio e fine stringa) di catturare ogni linea, non solo la prima

re.findall(r'\w+$', text, re.MULTILINE) 

['believe', 'Orion', 'Gate', 'rain', 'die']

### Classi

In [26]:
text = 'only 123 number, 456 pass! or else?'

In [27]:
# numbers

re.findall(r'\d+', text), re.findall(r'\D+', text)

(['123', '456'], ['only ', ' number, ', ' pass! or else?'])

In [28]:
# alphanumeric 

re.findall(r'\w+', text), re.findall(r'\W+', text)

(['only', '123', 'number', '456', 'pass', 'or', 'else'],
 [' ', ' ', ', ', ' ', '! ', ' ', '?'])

In [29]:
# whitespace

re.findall(r'\s+', text), re.findall(r'\S+', text)

([' ', ' ', ' ', ' ', ' ', ' '],
 ['only', '123', 'number,', '456', 'pass!', 'or', 'else?'])

## Backreference

In [30]:
# inizia e finisce con lo stesso carattere

re.match(r'(.)\d+(\1)', '1001'), re.match(r'(.)\d+(\1)', '1002')

(<re.Match object; span=(0, 4), match='1001'>, None)

In [31]:
# due volte la stessa parola

re.match(r'(\w+)(\1)', 'abbaabba')

<re.Match object; span=(0, 8), match='abbaabba'>

## Efficienza

In [32]:
%%time 
# veloce se appartiene 

re.match(r'(a+)+c', 'a' * 26 + 'c')

CPU times: user 64 μs, sys: 11 μs, total: 75 μs
Wall time: 78.7 μs


<re.Match object; span=(0, 27), match='aaaaaaaaaaaaaaaaaaaaaaaaaac'>

In [33]:
%%time
# mortale se non appartiene 

re.match(r'(a+)+c', 'a' * 26 + 'b')

CPU times: user 2.41 s, sys: 7.41 ms, total: 2.42 s
Wall time: 2.41 s
